# Evolutionary Crossover Mutation Testing Zone

In [ ]:
from evolutionary_prompt_embedding.argument_types import PooledPromptEmbedData
from evolutionary_prompt_embedding.image_creation import SDXLPromptEmbeddingImageCreator
from evolutionary_prompt_embedding.variation import \
    UniformGaussianMutatorArguments, PooledUniformGaussianMutator, PooledArithmeticCrossover, PooledUniformCrossover
from evolutionary_prompt_embedding.value_ranges import SDXLTurboEmbeddingRange, SDXLTurboPooledEmbeddingRange
from diffusers.utils import make_image_grid

In [ ]:
# These hold a prompt embedding range calculated by finding min/max range through parti prompts P2
embedding_range = SDXLTurboEmbeddingRange()
pooled_embedding_range = SDXLTurboPooledEmbeddingRange()
creator = SDXLPromptEmbeddingImageCreator(batch_size=1, inference_steps=4)

In [ ]:
# Create random image, average it with the artist prompt
init_crossover = PooledArithmeticCrossover(interpolation_weight=0.9, interpolation_weight_pooled=0.9)
artist1_arg = creator.arguments_from_prompt("van gogh") 
artist2_arg = creator.arguments_from_prompt("banksy")
result1_arg = init_crossover.crossover(artist1_arg,  
                                      PooledPromptEmbedData(embedding_range.random_tensor_in_range(), pooled_embedding_range.random_tensor_in_range()))
result2_arg = init_crossover.crossover(artist2_arg,  
                          PooledPromptEmbedData(embedding_range.random_tensor_in_range(), pooled_embedding_range.random_tensor_in_range()))
result1 = creator.create_solution(PooledPromptEmbedData(embedding_range.random_tensor_in_range(), pooled_embedding_range.random_tensor_in_range()))
result2 = creator.create_solution(PooledPromptEmbedData(embedding_range.random_tensor_in_range(), pooled_embedding_range.random_tensor_in_range()))
make_image_grid([result1.result.images[0], result2.result.images[0]], 1, 2)

In [ ]:
# Crossover
crossover = PooledArithmeticCrossover(interpolation_weight=0.1, interpolation_weight_pooled=0.1)
result_args = crossover.crossover(result1.arguments, result2.arguments)
result = creator.create_solution(result_args)
make_image_grid([result.result.images[0]], 1, 1)

In [ ]:
import torch
mutation_arguments = UniformGaussianMutatorArguments(mutation_rate=0.05, mutation_strength=1, 
                                                     clamp_range=(embedding_range.minimum, embedding_range.maximum)) 
mutation_arguments_pooled = UniformGaussianMutatorArguments(mutation_rate=0.05, mutation_strength=0.1, 
                                                            clamp_range=(pooled_embedding_range.minimum, pooled_embedding_range.maximum))
mutator = PooledUniformGaussianMutator(mutation_arguments, mutation_arguments_pooled)

prevargs = result.arguments
result_args = mutator.mutate(result.arguments)
result = creator.create_solution(result_args)
# print abs different in values 
print(torch.abs(result.arguments.prompt_embeds - prevargs.prompt_embeds).sum().item())
make_image_grid([result.result.images[0]], 1, 1)

In [ ]:
# Applying mutation continuously
import imageio

frames = [] 
mutation_arguments = UniformGaussianMutatorArguments(mutation_rate=0.05, mutation_strength=0.1, 
                                                     clamp_range=(embedding_range.minimum, embedding_range.maximum)) 
mutation_arguments_pooled = UniformGaussianMutatorArguments(mutation_rate=0.05, mutation_strength=0.05, 
                                                            clamp_range=(pooled_embedding_range.minimum, pooled_embedding_range.maximum))
mutator = PooledUniformGaussianMutator(mutation_arguments, mutation_arguments_pooled)

result = creator.create_solution(PooledPromptEmbedData(embedding_range.random_tensor_in_range(), pooled_embedding_range.random_tensor_in_range()))
for _ in range(4): 
    result_args = mutator.mutate(result.arguments) 
    result = creator.create_solution(result_args)  
    frames.append(result.result.images[0])
    
make_image_grid(frames, 1, len(frames))

# (Initial Tests without Library) Trying to perform crossover and mutation in latent space of images and text
Note: the tokenizer and text encoder have to be extracted from the diffusion pipeline in order
to create embeddings yourself. This is also dependent on the model you are using because they do not use
the same tokenizer and text encoder - some use multiple just like the SDXL models.

Here we are performing crossover and mutation with the prompt encodings.

In [ ]:
from evolutionary_model_helpers.auto_device import auto_to_device, auto_generator
from diffusers.utils import make_image_grid
from diffusers import DiffusionPipeline
import torch

In [ ]:
def setup_diffusion_pipeline(model_id, variant="fp16"):
    """
    Helper function to load a model from the HuggingFace model hub and return a pipeline.
    Tries to load the fp16 variant if available.
    """
    pipe = None
    try:
        pipe = DiffusionPipeline.from_pretrained(
            model_id, torch_dtype=torch.float16, variant=variant,
            use_safetensors=True, safety_checker=None, requires_safety_checker=False
        )
        pipe = auto_to_device(pipe)
        print(f"Loaded {pipe}")
    except Exception as e:
        print(f"Could not load {model_id}: {e}")

    return pipe

def uniform_gaussian_mutate_tensor(tensor, mutation_rate=0.05, mutation_strength=0.1, clamp_range=(-1, 1)):
    """
    Perform a uniform gaussian mutation on the tensor while keeping it on the same device.

    Args:
    - tensor (torch.Tensor): The tensor to mutate.
    - mutation_rate (float): Fraction of elements to mutate (between 0 and 1).
    - mutation_strength (float): The strength of the mutation, influencing how much each element can change.
    - clamp_range (tuple): A tuple of (min, max) to clamp the mutated values.

    Returns:
    - torch.Tensor: The mutated tensor.
    """
    device = tensor.device  # Get the device of the input tensor
    num_elements_to_mutate = int(torch.numel(tensor) * mutation_rate)
    indices_to_mutate = torch.randperm(torch.numel(tensor), device=device)[:num_elements_to_mutate]

    # Generate mutations
    mutations = torch.randn(num_elements_to_mutate, device=device) * mutation_strength
    flat_tensor = tensor.flatten()
    flat_tensor[indices_to_mutate] += mutations
    mutated_tensor = flat_tensor.view(tensor.shape)

    # Clamp values to ensure they remain within a reasonable range
    mutated_tensor = torch.clamp(mutated_tensor, min=clamp_range[0], max=clamp_range[1])
    
    return mutated_tensor

# Some examples for possible crossover and mutation in prompt encoding space

def uniform_crossover_tensors(tensor1, tensor2, crossover_rate=0.5):
    """
    Perform a uniform crossover operation between two tensors, assuming they are on the same device.

    Args:
    - tensor1 (torch.Tensor): The first parent tensor.
    - tensor2 (torch.Tensor): The second parent tensor.
    - crossover_rate (float): The rate at which elements from the second tensor are introduced into the first.

    Returns:
    - torch.Tensor: The resulting tensor after crossover.
    """
    if tensor1.shape != tensor2.shape:
        raise ValueError("Both tensors must have the same shape for crossover.")

    # Create a mask for crossover
    crossover_mask = torch.rand(tensor1.shape, device=tensor1.device) < crossover_rate

    # Perform crossover
    offspring = torch.where(crossover_mask, tensor2, tensor1)

    return offspring

def arithmetic_crossover(tensor1, tensor2, interpolation_weight=0.5):
    """
    Perform an interpolation-based crossover between two tensors.

    Args:
    - tensor1 (torch.Tensor): The first parent tensor.
    - tensor2 (torch.Tensor): The second parent tensor.
    - interpolation_weight (float): The weight for interpolation (between 0 and 1). A weight of 0.5 results in an
      equal blend of both tensors.

    Returns:
    - torch.Tensor: The resulting tensor after interpolation.
    """
    if tensor1.shape != tensor2.shape:
        raise ValueError("Both tensors must have the same shape for interpolation.")

    # Ensure tensors are on the same device
    device = tensor1.device
    tensor2 = tensor2.to(device)

    # Perform interpolation
    offspring = tensor1 * (1 - interpolation_weight) + tensor2 * interpolation_weight

    return offspring

models = [
    "stabilityai/sd-turbo",
    "stabilityai/sdxl-turbo",
    "stabilityai/stable-diffusion-xl-base-1.0",
    "stabilityai/stable-diffusion-2-1",
    "prompthero/openjourney",
    "kandinsky-community/kandinsky-3"
]

## Executing on SD Turbo

In [ ]:
def encode_text_sd(pipeline, device, prompt):
    tokenizer = pipeline.tokenizer
    text_encoder = pipeline.text_encoder
    
    text_inputs = tokenizer(
        prompt,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )

    # Get the output from the text encoder
    text_input_ids = text_inputs.input_ids
    untruncated_ids = tokenizer(prompt, padding="longest", return_tensors="pt").input_ids
    
    if untruncated_ids.shape[-1] >= text_input_ids.shape[-1] and not torch.equal(
        text_input_ids, untruncated_ids
    ):
        removed_text = tokenizer.batch_decode(untruncated_ids[:, tokenizer.model_max_length - 1 : -1])
        print(
            "The following part of your input was truncated because CLIP can only handle sequences up to"
            f" {tokenizer.model_max_length} tokens: {removed_text}"
        )
    
    with torch.no_grad():
        prompt_embeds = text_encoder(text_input_ids.to(device))
        prompt_embeds = prompt_embeds[0]
        
    return prompt_embeds

In [ ]:
pipeline = setup_diffusion_pipeline("stabilityai/sd-turbo")

In [ ]:
from evolutionary_model_helpers.auto_device import auto_device

# Your prompt
prompt = "hello dog"
batch_size = 2
device = auto_device()

prompt_embeds = encode_text_sd(pipeline, device, prompt)

# Use the pipeline with custom embeddings
images = pipeline(
    prompt_embeds=prompt_embeds,
    num_inference_steps=2,
    num_images_per_prompt=batch_size, 
    guidance_scale=0.0,
    generator=auto_generator(seed=1) # use generator for reproducibility
).images

make_image_grid(images, rows=1, cols=batch_size)

In [ ]:
# Now perform crossover
prompt_embeds_2 = encode_text_sd(pipeline, device, "big cat")
crossover_embeds = arithmetic_crossover(prompt_embeds, prompt_embeds_2, interpolation_weight=0.5)

images_after_crossover = pipeline(
    prompt_embeds=crossover_embeds,
    num_inference_steps=2,
    num_images_per_prompt=batch_size, 
    guidance_scale=0.0,
    generator=auto_generator(seed=1)
).images

make_image_grid(images_after_crossover, rows=1, cols=batch_size)

In [ ]:
# Now perform mutation
mutated_embeds = uniform_gaussian_mutate_tensor(crossover_embeds, mutation_rate=0.1, mutation_strength=0.5, clamp_range=(-20, 20))

images_after_crossover = pipeline(
    prompt_embeds=mutated_embeds,
    num_inference_steps=2,
    num_images_per_prompt=batch_size, 
    guidance_scale=0.0,
    generator=auto_generator(seed=1)
).images

make_image_grid(images_after_crossover, rows=1, cols=batch_size)

## Executing on SDXL Turbo

In [ ]:
# This code is taken from pipeline_stable_diffusion_xl from the diffusers library
def encode_text_sdxl(pipeline, device, prompt):
    tokenizer = pipeline.tokenizer
    tokenizer_2 = pipeline.tokenizer_2
    text_encoder = pipeline.text_encoder
    text_encoder_2 = pipeline.text_encoder_2
    
    tokenizers = [tokenizer, tokenizer_2] if tokenizer is not None else [tokenizer_2]
    text_encoders = ([text_encoder, text_encoder_2] if text_encoder is not None else [text_encoder_2])
    
    # We only use one prompt here, but you could also use two prompts for SDXL
    prompt_2 = prompt
    prompt_2 = [prompt_2] if isinstance(prompt_2, str) else prompt_2

    # textual inversion: procecss multi-vector tokens if necessary
    prompt_embeds_list = []
    prompts = [prompt, prompt_2]
    
    for prompt, tokenizer, text_encoder in zip(prompts, tokenizers, text_encoders):
        text_inputs = tokenizer(
            prompt,
            padding="max_length",
            max_length=tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        )
    
        text_input_ids = text_inputs.input_ids
        untruncated_ids = tokenizer(prompt, padding="longest", return_tensors="pt").input_ids
    
        if untruncated_ids.shape[-1] >= text_input_ids.shape[-1] and not torch.equal(
            text_input_ids, untruncated_ids
        ):
            removed_text = tokenizer.batch_decode(untruncated_ids[:, tokenizer.model_max_length - 1 : -1])
            print(
                "The following part of your input was truncated because CLIP can only handle sequences up to"
                f" {tokenizer.model_max_length} tokens: {removed_text}"
            )
    
        prompt_embeds = text_encoder(text_input_ids.to(device), output_hidden_states=True)
    
        # We are only ALWAYS interested in the pooled output of the final text encoder
        pooled_prompt_embeds = prompt_embeds[0]
        # "2" because SDXL always indexes from the penultimate layer.
        prompt_embeds = prompt_embeds.hidden_states[-2]
        prompt_embeds_list.append(prompt_embeds)
    # end for
    prompt_embeds = torch.concat(prompt_embeds_list, dim=-1)
    return (prompt_embeds, pooled_prompt_embeds)

In [ ]:
pipeline = setup_diffusion_pipeline("stabilityai/sdxl-turbo")

In [ ]:
prompt = "a dog"
batch_size = 2
device = auto_device()

prompt_embeds, pooled_prompt_embeds = encode_text_sdxl(pipeline, device, prompt)

images = pipeline(
    prompt_embeds=prompt_embeds,
    pooled_prompt_embeds=pooled_prompt_embeds,
    num_inference_steps=3,
    num_images_per_prompt=batch_size, #defined by prompt_embeds
    guidance_scale=0.0,
    generator=auto_generator(seed=1)
).images

make_image_grid(images, rows=1, cols=batch_size)

In [ ]:
# Now perform crossover, the other kind
prompt_embeds_2, pooled_prompt_embeds_2 = encode_text_sdxl(pipeline, device, "a cat")
crossover_embeds = uniform_crossover_tensors(prompt_embeds, prompt_embeds_2, crossover_rate=0.5)

images_after_crossover = pipeline(
    prompt_embeds=crossover_embeds,
    pooled_prompt_embeds=pooled_prompt_embeds,
    num_inference_steps=3,
    num_images_per_prompt=batch_size, #defined by prompt_embeds
    guidance_scale=0.0,
    generator=auto_generator(seed=1)
).images

make_image_grid(images_after_crossover, rows=1, cols=batch_size)

In [ ]:
# Now perform mutation
mutated_embeds = uniform_gaussian_mutate_tensor(crossover_embeds, mutation_rate=0.1, mutation_strength=0.5, clamp_range=(-10, 10))

images_after_crossover = pipeline(
    prompt_embeds=mutated_embeds,
    pooled_prompt_embeds=pooled_prompt_embeds,
    num_inference_steps=3,
    num_images_per_prompt=batch_size, #defined by prompt_embeds
    guidance_scale=0.0,
    generator=auto_generator(seed=1)
).images

make_image_grid(images_after_crossover, rows=1, cols=batch_size)